In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import string
import wordcloud

In [2]:
df = pd.read_csv('ready_to_go.csv')
df_test = pd.read_csv('ready_to_go_validate.csv')

In [3]:
df_test['Comments'] = df_test['Comments'].fillna('')

In [4]:
df_set = pd.concat([df,df_test],ignore_index=True)

In [5]:
df_set

,target,Comments,Tokenize_review
0,Positive,coming the borders and will kill you all,"['coming', 'borders', 'kill']"
1,Positive,getting borderlands and will kill you all,"['getting', 'borderlands', 'kill']"
2,Positive,coming borderlands and will murder you all,"['coming', 'borderlands', 'murder']"
3,Positive,getting borderlands and will murder you all,"['getting', 'borderlands', 'murder']"
4,Positive,getting into borderlands and can murder you all,"['getting', 'borderlands', 'murder']"
...,...,...,...
68318,Irrelevant,toronto the arts and culture capital canada wo...,"['toronto', 'arts', 'culture', 'capital', 'can..."
68319,Irrelevant,this actually good move to bring more viewer w...,"['actually', 'good', 'move', 'bring', 'viewer'..."
68320,Positive,today sucked time drink wine play borderlands ...,"['today', 'sucked', 'time', 'drink', 'wine', '..."
68321,Positive,bought fraction microsoft today small wins,"['bought', 'fraction', 'microsoft', 'today', '..."


In [7]:
import pandas as pd
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),
        min_df=2,          # 👈 relaxed
        sublinear_tf=True,
        norm='l2'
    )),
    ('nb', MultinomialNB(alpha=0.001))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    pipeline,
    df_set['Comments'],   # raw text
    df_set['target'],     # ⚠️ use original labels
    cv=skf,
    scoring='accuracy'
)

print("NB Mean CV Accuracy:", cv_scores.mean())


NB Mean CV Accuracy: 0.9431377277116961


In [9]:
alphas = [0.0001, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.3, 0.5, 1]

In [10]:
import numpy as np
import pandas as pd

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score


In [11]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [12]:
results = []

for a in alphas:
    pipeline_nb = Pipeline([
        ('tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,          # CV-safe
            sublinear_tf=True,
            norm='l2'
        )),
        ('nb', MultinomialNB(alpha=a))
    ])
    
    cv_acc = cross_val_score(
        pipeline_nb,
        df_set['Comments'],
        df_set['target'],
        cv=skf,
        scoring='accuracy'
    )
    
    cv_f1 = cross_val_score(
        pipeline_nb,
        df_set['Comments'],
        df_set['target'],
        cv=skf,
        scoring='f1_weighted'
    )
    
    results.append({
        "Model": "MultinomialNB",
        "Alpha": a,
        "CV_Accuracy_Mean": cv_acc.mean(),
        "CV_Accuracy_Std": cv_acc.std(),
        "CV_F1_Mean": cv_f1.mean(),
        "CV_F1_Std": cv_f1.std()
    })


In [13]:
results_df = pd.DataFrame(results)
results_df.sort_values("CV_F1_Mean", ascending=False)


,Model,Alpha,CV_Accuracy_Mean,CV_Accuracy_Std,CV_F1_Mean,CV_F1_Std
1,MultinomialNB,0.0010,0.943138,0.001587,0.943109,0.001581
0,MultinomialNB,0.0001,0.942962,0.001783,0.942933,0.001779
2,MultinomialNB,0.0050,0.942933,0.001468,0.942904,0.001457
3,MultinomialNB,0.0100,0.942669,0.001361,0.942641,0.001351
4,MultinomialNB,0.0200,0.942040,0.001680,0.942008,0.001668
5,MultinomialNB,0.0500,0.940123,0.001416,0.940087,0.001402
6,MultinomialNB,0.1000,0.937152,0.001287,0.937103,0.001281
7,MultinomialNB,0.3000,0.920320,0.001191,0.920139,0.001191
8,MultinomialNB,0.5000,0.899990,0.001221,0.899350,0.001210
9,MultinomialNB,1.0000,0.846435,0.002044,0.842721,0.002033


In [14]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),
        min_df=2,          # 👈 relaxed
        sublinear_tf=True,
        norm='l2'
    )),
    ('nb', MultinomialNB(alpha=0.0010))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    pipeline,
    df_set['Comments'],   # raw text
    df_set['target'],     # ⚠️ use original labels
    cv=skf,
    scoring='accuracy'
)
print("NB Mean CV Accuracy:", cv_scores.mean())

NB Mean CV Accuracy: 0.9431377277116961


In [15]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import joblib

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),
        min_df=2,
        sublinear_tf=True,
        norm='l2'
    )),
    ('nb', MultinomialNB(alpha=0.0010))
])

# Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    pipeline,
    df_set['Comments'],
    df_set['target'],
    cv=skf,
    scoring='accuracy'
)

print("NB Mean CV Accuracy:", cv_scores.mean())

# Train on the full dataset
pipeline.fit(df_set['Comments'], df_set['target'])

# Save the trained pipeline
joblib.dump(pipeline, "multinomial_nb_pipeline.pkl")

print("Model saved successfully!")

NB Mean CV Accuracy: 0.9431377277116961
Model saved successfully!


In [16]:
import joblib

model = joblib.load("multinomial_nb_pipeline.pkl")

In [17]:
text = ["This product is amazing!"]

prediction = model.predict(text)
print(prediction)

# Prediction probabilities (optional)
prob = model.predict_proba(text)
print(prob)

['Irrelevant']
[[0.55463414 0.00383573 0.43053505 0.01099508]]
